In [2]:
import OpenImageIO as oiio
import numpy as np

from colour.models.rgb.datasets.arri import MATRIX_ARRI_WIDE_GAMUT_4_TO_XYZ
from colour.models.rgb.datasets.itur_bt_709 import MATRIX_XYZ_TO_BT709

from pcm.paths import IMAGES_ROOT

In [3]:
LIN_AWG4_PATH = IMAGES_ROOT / "multiple/ARRI_Helen_John_ALEXA_35_AWG4_1K.exr"
LIN_BT709_PATH = IMAGES_ROOT / "multiple/ARRI_Helen_John_ALEXA_35_lin_BT709_1K.exr"

In [4]:
def read(lin_awg4_img_path) -> np.ndarray:
    input_image = oiio.ImageInput.open(lin_awg4_img_path.as_posix())
    if input_image is None:
        raise RuntimeError("Failed to open image: " + oiio.geterror())
    spec = input_image.spec()
    if spec.nchannels < 3:
        raise ValueError("Image has fewer than 3 channels.")
    image_data = input_image.read_image(format=oiio.FLOAT)
    if image_data is None:
        raise RuntimeError("Failed to read image: " + input_image.geterror())
    lin_awg4_array = np.asarray(image_data).reshape(spec.height, spec.width, spec.nchannels)
    return lin_awg4_array

def convert_to_bt709(lin_awg4_array: np.ndarray) -> np.ndarray:
    lin_xyz_array = lin_awg4_array[..., :3] @ MATRIX_ARRI_WIDE_GAMUT_4_TO_XYZ.T
    lin_bt709_array = lin_xyz_array @ MATRIX_XYZ_TO_BT709.T
    return lin_bt709_array

def write(lin_bt709_array: np.ndarray, lin_bt709_img_path) -> None:
    output_spec = oiio.ImageSpec(lin_bt709_array.shape[1], lin_bt709_array.shape[0], 3, oiio.FLOAT)
    output_image = oiio.ImageOutput.create(lin_bt709_img_path.as_posix())
    if output_image is None:
        raise RuntimeError("Failed to create output image: " + oiio.geterror())
    if not output_image.open(lin_bt709_img_path.as_posix(), output_spec):
        raise RuntimeError("Failed to open output image: " + output_image.geterror())
    if not output_image.write_image(lin_bt709_array):
        raise RuntimeError("Failed to write image: " + output_image.geterror())
    output_image.close()


In [5]:
def convert_helen_and_john():
    lin_awg4_array = read(LIN_AWG4_PATH)
    print(f"read: {LIN_AWG4_PATH}")
    lin_bt709_array = convert_to_bt709(lin_awg4_array)
    write(lin_bt709_array, LIN_BT709_PATH)
    print(f"wrote: {LIN_BT709_PATH}")

In [5]:
convert_helen_and_john()

read: /usr/local/repos/git/jgoldstone/pcm-dev/resources/images/multiple/ARRI_Helen_John_ALEXA_35_AWG4.exr
wrote: /usr/local/repos/git/jgoldstone/pcm-dev/resources/images/multiple/ARRI_Helen_John_ALEXA_35_lin_BT709.exr
